# Base de dados municipais — IBGE

## Objetivo

Este notebook reúne informações básicas dos municípios brasileiros a partir de dados disponibilizados pelo Instituto Brasileiro de Geografia e Estatística (IBGE).

A base final contém uma linha para cada município, identificado pelo código oficial do IBGE, e reúne indicadores de diferentes áreas, como população, território, economia, educação, saúde e infraestrutura.

## Fonte dos dados

Os dados foram obtidos por meio de APIs oficiais do IBGE:

- **API de Localidades:** utilizada para obter a relação dos municípios, seus códigos IBGE, unidades da federação, estados e regiões.
- **API de indicadores do Portal Cidades e Estados (Cidades@):** utilizada para consultar os indicadores municipais disponibilizados pelo IBGE.

A relação de municípios é obtida pela API de Localidades:

`https://servicodados.ibge.gov.br/api/v1/localidades/municipios`

Os indicadores são consultados por unidade da federação por meio da API do Cidades@.

## Abrangência

A base contempla os **5.570 municípios brasileiros** existentes no momento da consulta.

O **código IBGE** é utilizado como chave para relacionar os municípios das diferentes fontes.

## Data de referência dos dados

Os indicadores não têm, necessariamente, o mesmo ano de referência. Cada variável utiliza o período mais recente disponibilizado pelo IBGE para aquele indicador no momento da consulta.

Entre as referências utilizadas estão:

- **População do Censo:** Censo Demográfico 2022;
- **População estimada:** estimativa populacional de 2026;
- **Área territorial:** referência de 2025;
- **PIB per capita:** ano de referência 2023;
- **IDHM:** ano de referência 2010.

Outros indicadores podem ter anos de referência diferentes, de acordo com a série estatística correspondente.

Quando essa informação está disponível, as variáveis temporais são acompanhadas por uma coluna com o respectivo **ano de referência**. Isso permite identificar o período ao qual cada valor corresponde e evita tratar todos os indicadores como se fossem do mesmo ano.

## Validação

Depois da coleta e consolidação dos dados, alguns valores foram conferidos manualmente no portal oficial **IBGE — Cidades e Estados**:

[https://www.ibge.gov.br/cidades-e-estados.html](https://www.ibge.gov.br/cidades-e-estados.html)

Foram selecionados municípios para comparar os valores exibidos no portal com aqueles obtidos pelo notebook.

Os valores conferidos foram compatíveis com os dados retornados pela API. Essa verificação serviu como uma validação adicional do processo de consulta, transformação e associação dos dados.

A conferência manual não representa uma auditoria de todos os municípios e indicadores, mas ajuda a verificar se o procedimento de coleta e integração está funcionando corretamente.

## Observação sobre o código IBGE

A API de Localidades utiliza o código oficial de **7 dígitos** para identificar os municípios. Já os resultados da API de indicadores do Cidades@ podem apresentar o código municipal em formato de **6 dígitos**.

O notebook trata essa diferença antes de fazer a integração entre as fontes. No resultado final, é mantido o **código IBGE oficial de 7 dígitos**.

## Resultado

Ao final do processamento, é gerado um DataFrame com os **5.570 municípios brasileiros** e os indicadores disponíveis nas fontes consultadas.

A base também pode ser complementada posteriormente com outras bases oficiais do IBGE. Uma possibilidade é incorporar a base específica do **Produto Interno Bruto dos Municípios**, incluindo o PIB total e séries históricas de indicadores econômicos.

In [1]:
# ============================================================
# 1. BIBLIOTECAS
# ============================================================

import requests
import pandas as pd
import numpy as np
import time
import json
from io import BytesIO

print("Bibliotecas carregadas.")

Bibliotecas carregadas.


In [2]:
# ============================================================
# 2. OBTER OS MUNICÍPIOS PELA API OFICIAL DO IBGE
# ============================================================

from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

URL_MUNICIPIOS = (
    "https://servicodados.ibge.gov.br/api/v1/localidades/municipios"
)

# Configurar sessão com novas tentativas automáticas
session = requests.Session()

retry_strategy = Retry(
    total=5,
    connect=5,
    read=3,
    backoff_factor=2,
    status_forcelist=[429, 500, 502, 503, 504],
    allowed_methods=["GET"]
)

adapter = HTTPAdapter(max_retries=retry_strategy)

session.mount("https://", adapter)
session.mount("http://", adapter)

# Consultar API
try:
    response = session.get(
        URL_MUNICIPIOS,
        timeout=(30, 120)
    )

    response.raise_for_status()

    municipios_raw = response.json()

    print(f"Municípios retornados pela API: {len(municipios_raw):,}")

except requests.exceptions.RequestException as e:
    raise RuntimeError(
        f"Falha ao consultar a API de Localidades do IBGE: {e}"
    )

Municípios retornados pela API: 5,571


In [3]:
# ============================================================
# 3. ORGANIZAR MUNICÍPIOS
# ============================================================

municipios = []

for m in municipios_raw:

    try:
        uf = m["microrregiao"]["mesorregiao"]["UF"]

        municipios.append({
            "codigo_ibge": m["id"],
            "municipio": m["nome"],
            "uf": uf["sigla"],
            "codigo_uf": uf["id"],
            "estado": uf["nome"],
            "regiao": uf["regiao"]["nome"],
            "codigo_regiao": uf["regiao"]["id"]
        })

    except Exception:
        # Algumas localidades podem ter estrutura diferente
        pass

df_municipios = pd.DataFrame(municipios)

print(f"Registros obtidos: {len(df_municipios):,}")
print(f"Códigos IBGE únicos: {df_municipios.codigo_ibge.nunique():,}")

Registros obtidos: 5,570
Códigos IBGE únicos: 5,570


In [4]:
# ============================================================
# 4. INDICADORES DA API CIDADES@
# ============================================================

INDICADORES = {
    29167: "area_km2",
    96386: "densidade_demografica",
    29171: "populacao_estimada",
    96385: "populacao_censo",
    47001: "pib_per_capita",
    30255: "idhm",
    60045: "taxa_escolarizacao_6_14",
    30279: "mortalidade_infantil",
    28141: "receitas_brutas_realizadas",
    29749: "despesas_brutas_empenhadas",
    143514: "pessoal_ocupado",
    143558: "rendimento_medio_sal_min",
    60036: "populacao_ocupada",
    60037: "populacao_renda_per_capita_maior_meio_salario",
    60048: "transferencias_correntes",
    60030: "esgotamento_sanitario_adequado",
    60031: "urbanizacao_vias_publicas",
    60029: "arborizacao_vias_publicas",
    28242: "estabelecimentos_saude_sus",
    5908: "matriculas_ensino_fundamental",
    5913: "matriculas_ensino_medio",
    5950: "estabelecimentos_ensino_fundamental",
    5955: "estabelecimentos_ensino_medio",
    5929: "docentes_ensino_fundamental",
    5934: "docentes_ensino_medio",
    95335: "area_urbanizada_km2",
    60032: "internacoes_diarreia_sus_100mil",
    93371: "populacao_exposta_risco",
    77861: "bioma",
    87529: "hierarquia_urbana",
    87530: "regiao_influencia",
    91245: "regiao_intermediaria",
    91247: "regiao_imediata",
    91249: "mesorregiao",
    91251: "microrregiao",
    82270: "sistema_costeiro_marinho",
}

print(f"Total de indicadores selecionados: {len(INDICADORES)}")

Total de indicadores selecionados: 36


In [5]:
# ============================================================
# 5. CONSULTA À API CIDADES@
# ============================================================

indicadores_str = "|".join(str(x) for x in INDICADORES.keys())

URL_BASE = (
    "https://servicodados.ibge.gov.br/api/v1/pesquisas/"
    f"indicadores/{indicadores_str}/resultados"
)

In [6]:
# ============================================================
# 6. FUNÇÃO PARA CONSULTAR UMA UF
# ============================================================

def consultar_uf(uf_codigo):

    url = f"{URL_BASE}/{uf_codigo}xxxxx"

    response = requests.get(url, timeout=120)
    response.raise_for_status()

    return response.json()

In [7]:
# ============================================================
# 7. BAIXAR OS DADOS
# ============================================================

# Código IBGE das UFs
codigos_uf = (
    df_municipios[["uf", "codigo_uf"]]
    .drop_duplicates()
    .sort_values("codigo_uf")
)

dados_api = []

for _, row in codigos_uf.iterrows():

    uf = row["uf"]
    codigo_uf = int(row["codigo_uf"])

    print(f"Consultando {uf} ({codigo_uf})...", end=" ")

    try:

        resultado = consultar_uf(codigo_uf)

        dados_api.extend(resultado)

        print(f"OK — {len(resultado)} registros")

    except Exception as e:

        print(f"ERRO: {e}")

    time.sleep(0.3)

print()
print(f"Total de registros retornados: {len(dados_api):,}")

Consultando RO (11)... OK — 35 registros
Consultando AC (12)... OK — 35 registros
Consultando AM (13)... OK — 35 registros
Consultando RR (14)... OK — 35 registros
Consultando PA (15)... OK — 35 registros
Consultando AP (16)... OK — 35 registros
Consultando TO (17)... OK — 35 registros
Consultando MA (21)... OK — 35 registros
Consultando PI (22)... OK — 35 registros
Consultando CE (23)... OK — 35 registros
Consultando RN (24)... OK — 35 registros
Consultando PB (25)... OK — 35 registros
Consultando PE (26)... OK — 35 registros
Consultando AL (27)... OK — 35 registros
Consultando SE (28)... OK — 35 registros
Consultando BA (29)... OK — 35 registros
Consultando MG (31)... OK — 35 registros
Consultando ES (32)... OK — 35 registros
Consultando RJ (33)... OK — 35 registros
Consultando SP (35)... OK — 35 registros
Consultando PR (41)... OK — 35 registros
Consultando SC (42)... OK — 35 registros
Consultando RS (43)... OK — 35 registros
Consultando MS (50)... OK — 35 registros
Consultando MT (

In [8]:
# ============================================================
# 8. TRANSFORMAR O RETORNO DA API
# ============================================================

# A estrutura real é:
#
# [
#   {
#       "id": CODIGO_INDICADOR,
#       "res": [
#           {
#               "localidade": "CODIGO_MUNICIPIO",
#               "res": {
#                   "ANO": "VALOR"
#               }
#           }
#       ]
#   }
# ]
#
# Portanto precisamos inverter essa estrutura para:
#
# municipio -> indicador -> valor

registros = {}

for indicador_data in dados_api:

    codigo_indicador = indicador_data.get("id")

    if codigo_indicador not in INDICADORES:
        continue

    nome_indicador = INDICADORES[codigo_indicador]

    municipios_indicador = indicador_data.get("res", [])

    for municipio_data in municipios_indicador:

        codigo_municipio = municipio_data.get("localidade")

        if codigo_municipio is None:
            continue

        codigo_municipio = int(codigo_municipio)

        if codigo_municipio not in registros:
            registros[codigo_municipio] = {
                "codigo_ibge": codigo_municipio
            }

        valores = municipio_data.get("res", {})

        if not isinstance(valores, dict) or not valores:
            continue

        # ----------------------------------------------------
        # Selecionar o ano mais recente disponível
        # ----------------------------------------------------

        anos_validos = []

        for ano in valores.keys():

            try:
                anos_validos.append(int(ano))
            except:
                pass

        if not anos_validos:
            continue

        ano_mais_recente = max(anos_validos)

        valor = valores[str(ano_mais_recente)]

        registros[codigo_municipio][nome_indicador] = valor
        registros[codigo_municipio][f"{nome_indicador}_ano"] = ano_mais_recente


df_indicadores = pd.DataFrame(
    list(registros.values())
)

print("=" * 60)
print("RESULTADO DA EXTRAÇÃO")
print("=" * 60)

print(
    f"Municípios encontrados: "
    f"{df_indicadores['codigo_ibge'].nunique():,}"
)

print(
    f"Colunas de indicadores: "
    f"{len(df_indicadores.columns):,}"
)

print(
    f"Linhas: "
    f"{len(df_indicadores):,}"
)

display(df_indicadores.head())

RESULTADO DA EXTRAÇÃO
Municípios encontrados: 5,571
Colunas de indicadores: 71
Linhas: 5,571


,codigo_ibge,estabelecimentos_saude_sus,estabelecimentos_saude_sus_ano,mortalidade_infantil,mortalidade_infantil_ano,matriculas_ensino_fundamental,matriculas_ensino_fundamental_ano,matriculas_ensino_medio,matriculas_ensino_medio_ano,docentes_ensino_fundamental,...,microrregiao,microrregiao_ano,area_urbanizada_km2,area_urbanizada_km2_ano,densidade_demografica,densidade_demografica_ano,pessoal_ocupado,pessoal_ocupado_ano,rendimento_medio_sal_min,rendimento_medio_sal_min_ano
0,110001,7,2009.0,2.54,2025,2938,2025,1019,2025,197,...,Cacoal,2022.0,7.15,2022.0,3.04,2022.0,3855,2024.0,1.8,2024.0
1,110002,16,2009.0,11.08,2025,13009,2025,4451,2025,525,...,Ariquemes,2022.0,32.13,2022.0,21.88,2022.0,25853,2024.0,2.1,2024.0
2,110003,6,2009.0,12.35,2025,669,2025,226,2025,50,...,Colorado do Oeste,2022.0,2.19,2022.0,4.07,2022.0,856,2024.0,2.2,2024.0
3,110004,14,2009.0,13.91,2025,11187,2025,4558,2025,551,...,Cacoal,2022.0,28.25,2022.0,22.91,2022.0,26186,2024.0,2.0,2024.0
4,110005,7,2009.0,16.67,2025,2264,2025,631,2025,95,...,Colorado do Oeste,2022.0,6.36,2022.0,5.71,2022.0,3707,2024.0,2.1,2024.0


In [9]:
# ============================================================
# 9. INDICADORES OBTIDOS
# ============================================================

print("Colunas disponíveis:")
print()

for coluna in df_indicadores.columns:
    print(coluna)

Colunas disponíveis:

codigo_ibge
estabelecimentos_saude_sus
estabelecimentos_saude_sus_ano
mortalidade_infantil
mortalidade_infantil_ano
matriculas_ensino_fundamental
matriculas_ensino_fundamental_ano
matriculas_ensino_medio
matriculas_ensino_medio_ano
docentes_ensino_fundamental
docentes_ensino_fundamental_ano
docentes_ensino_medio
docentes_ensino_medio_ano
estabelecimentos_ensino_fundamental
estabelecimentos_ensino_fundamental_ano
estabelecimentos_ensino_medio
estabelecimentos_ensino_medio_ano
pib_per_capita
pib_per_capita_ano
arborizacao_vias_publicas
arborizacao_vias_publicas_ano
esgotamento_sanitario_adequado
esgotamento_sanitario_adequado_ano
urbanizacao_vias_publicas
urbanizacao_vias_publicas_ano
populacao_renda_per_capita_maior_meio_salario
populacao_renda_per_capita_maior_meio_salario_ano
taxa_escolarizacao_6_14
taxa_escolarizacao_6_14_ano
populacao_exposta_risco
populacao_exposta_risco_ano
populacao_censo
populacao_censo_ano
populacao_estimada
populacao_estimada_ano
internac

In [10]:
# ============================================================
# 10. CORRIGIR CÓDIGO IBGE E FAZER A JUNÇÃO
# ============================================================

# A API Cidades@ usa código de município com 6 dígitos.
#
# Exemplo:
# API Localidades:
#     5208707 = Goiânia
#
# API Cidades@:
#     520870
#
# Portanto, vamos criar uma chave de 6 dígitos nas duas bases.

df_municipios["codigo_ibge_6"] = (
    df_municipios["codigo_ibge"] // 10
)

df_indicadores["codigo_ibge_6"] = (
    df_indicadores["codigo_ibge"].astype(str).str.zfill(6)
)

# Converter para inteiro para garantir compatibilidade
df_indicadores["codigo_ibge_6"] = pd.to_numeric(
    df_indicadores["codigo_ibge_6"],
    errors="coerce"
).astype("Int64")


# ------------------------------------------------------------
# Verificar correspondência
# ------------------------------------------------------------

codigos_municipios = set(
    df_municipios["codigo_ibge_6"].dropna()
)

codigos_indicadores = set(
    df_indicadores["codigo_ibge_6"].dropna()
)


# ------------------------------------------------------------
# Fazer a junção
# ------------------------------------------------------------

df_final = df_municipios.merge(
    df_indicadores.drop(
        columns=["codigo_ibge"],
        errors="ignore"
    ),
    on="codigo_ibge_6",
    how="left"
)

# Remover chave auxiliar
df_final = df_final.drop(
    columns=["codigo_ibge_6"]
)

print()
print("Dimensão final:")
print(df_final.shape)

display(df_final.head())


Dimensão final:
(5570, 77)


,codigo_ibge,municipio,uf,codigo_uf,estado,regiao,codigo_regiao,estabelecimentos_saude_sus,estabelecimentos_saude_sus_ano,mortalidade_infantil,...,microrregiao,microrregiao_ano,area_urbanizada_km2,area_urbanizada_km2_ano,densidade_demografica,densidade_demografica_ano,pessoal_ocupado,pessoal_ocupado_ano,rendimento_medio_sal_min,rendimento_medio_sal_min_ano
0,1100015,Alta Floresta D'Oeste,RO,11,Rondônia,Norte,1,7,2009.0,2.54,...,Cacoal,2022.0,7.15,2022.0,3.04,2022.0,3855,2024.0,1.8,2024.0
1,1100023,Ariquemes,RO,11,Rondônia,Norte,1,16,2009.0,11.08,...,Ariquemes,2022.0,32.13,2022.0,21.88,2022.0,25853,2024.0,2.1,2024.0
2,1100031,Cabixi,RO,11,Rondônia,Norte,1,6,2009.0,12.35,...,Colorado do Oeste,2022.0,2.19,2022.0,4.07,2022.0,856,2024.0,2.2,2024.0
3,1100049,Cacoal,RO,11,Rondônia,Norte,1,14,2009.0,13.91,...,Cacoal,2022.0,28.25,2022.0,22.91,2022.0,26186,2024.0,2.0,2024.0
4,1100056,Cerejeiras,RO,11,Rondônia,Norte,1,7,2009.0,16.67,...,Colorado do Oeste,2022.0,6.36,2022.0,5.71,2022.0,3707,2024.0,2.1,2024.0


In [11]:
# ============================================================
# AUDITORIA DA JUNÇÃO
# ============================================================

print("=" * 60)
print("AUDITORIA")
print("=" * 60)

print(
    f"Municípios na API de localidades: "
    f"{df_municipios['codigo_ibge'].nunique():,}"
)

print(
    f"Municípios na base final: "
    f"{df_final['codigo_ibge'].nunique():,}"
)

# Quantos municípios possuem pelo menos um indicador?
colunas_indicadores = [
    c for c in df_indicadores.columns
    if c not in ["codigo_ibge", "codigo_ibge_6"]
]

# Apenas colunas que realmente estão no resultado
colunas_indicadores = [
    c for c in colunas_indicadores
    if c in df_final.columns
]

tem_indicador = df_final[colunas_indicadores].notna().any(axis=1)

print(
    f"Municípios com pelo menos um indicador: "
    f"{tem_indicador.sum():,}"
)

print(
    f"Municípios sem nenhum indicador: "
    f"{(~tem_indicador).sum():,}"
)

AUDITORIA
Municípios na API de localidades: 5,570
Municípios na base final: 5,570
Municípios com pelo menos um indicador: 5,570
Municípios sem nenhum indicador: 0


In [12]:
# Mostrar um município com TODAS as colunas
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 40)

display(
    df_final[df_final["codigo_ibge"] == 3504305].T
)

,3316
codigo_ibge,3504305
municipio,Avaí
uf,SP
codigo_uf,35
estado,São Paulo
regiao,Sudeste
codigo_regiao,3
estabelecimentos_saude_sus,1
estabelecimentos_saude_sus_ano,2009.0
mortalidade_infantil,-


In [13]:
# ============================================================
# 15. ORGANIZAÇÃO FINAL
# ============================================================

colunas_base = [
    "codigo_ibge",
    "municipio",
    "uf",
    "codigo_uf",
    "estado",
    "regiao",
    "codigo_regiao"
]

outras_colunas = [
    c for c in df_final.columns
    if c not in colunas_base
]

df_final = df_final[
    colunas_base + sorted(outras_colunas)
]

df_final = df_final.sort_values(
    ["uf", "municipio"]
).reset_index(drop=True)


In [14]:
# ============================================================
# 17. EXPORTAR CSV
# ============================================================

arquivo_csv = "municipios_ibge.csv"

df_final.to_csv(
    arquivo_csv,
    index=False,
    encoding="utf-8-sig"
)

print(f"Arquivo criado: {arquivo_csv}")
print(f"Linhas: {len(df_final):,}")
print(f"Colunas: {len(df_final.columns):,}")

Arquivo criado: municipios_ibge.csv
Linhas: 5,570
Colunas: 77
